# 🎙 Транскрибация видео с разделением по спикерам (Colab + Google Drive)

Кладёшь видео в папку на Google Drive → запускаешь ячейки сверху вниз → в той же папке
появляются готовые транскрипты (.docx и .txt): «Спикер 1:», «Спикер 2:» и абзацы с таймкодами —
тот же формат, что отдаёт SpeechToText.

Под капотом: WhisperX **large-v3** (транскрибация + чистка галлюцинаций + повторный проход
по проблемным зонам) и **pyannote speaker-diarization-community-1** (диаризация).
Проект самостоятельный — только транскрипция, ничего лишнего.

---

## Разовая настройка (5 минут, один раз)

1. **GPU:** меню «Среда выполнения → Сменить среду выполнения» → выбери **T4 GPU**.
2. **Секреты** (значок ключа 🔑 на левой панели, у каждого включи «Доступ из блокнота»):
   - `HF_TOKEN` — токен Hugging Face ([создать](https://huggingface.co/settings/tokens), тип Read) — нужен для диаризации;
   - `GITHUB_TOKEN` — GitHub Personal Access Token с доступом на чтение к приватному репозиторию
     `Hipposum/Hipposum-colab-drive-transcriber` (Settings → Developer settings → Fine-grained tokens → Contents: Read).
3. **Условия моделей диаризации:** зайди под своим HF-аккаунтом и нажми «Agree» на страницах:
   - [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) (основная)
   - [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) и
     [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) (фолбэк)
4. **Папка на Google Drive:** создай папку (по умолчанию `Transcribe` в корне Диска) и закинь туда видео.

Дальше — просто «Среда выполнения → Выполнить всё». Скорость на T4: примерно 5–8 минут на час видео
без диаризации и 10–15 минут с ней.

## ⚙️ Настройки запуска
**Меняй только эту ячейку.**

In [ ]:
# ══════════════════════════════════════════════════════════
#  НАСТРОЙКИ ЗАПУСКА
# ══════════════════════════════════════════════════════════

# Папка с видео на Google Drive (путь после MyDrive/).
# Обрабатываются все вложенные подпапки. Форматы: mp4, mov, avi, mkv, webm, m4v, zip.
DRIVE_FOLDER = "Transcribe"

# Разделение по спикерам (True/False). Без него — в ~2 раза быстрее, но реплики не размечены.
DIARIZE = True

# Сколько спикеров ожидается в записи (диапазон). Если не знаешь — оставь как есть.
MIN_SPEAKERS = 1
MAX_SPEAKERS = 6

# Язык речи ("ru", "en", ...)
LANGUAGE = "ru"

# Сколько видео обработать за один запуск (0 = все найденные)
MAX_VIDEOS = 0

# Пропускать видео короче N секунд
MIN_DURATION_SEC = 30

# Подсказка Whisper — ТОЛЬКО ключевые слова темы через запятую, НЕ предложения
# (полные предложения Whisper повторяет при тишине → галлюцинации). "" = без подсказки.
INITIAL_PROMPT = ""

# Модель Whisper: "large-v3" (лучшее качество) | "medium" | "small" (быстрее, хуже)
WHISPER_MODEL = "large-v3"

print("✅ Настройки заданы. Запускай следующие ячейки.")

## 📦 Установка зависимостей и код проекта (~3–4 минуты)

In [ ]:
import subprocess, shutil, os, sys

# Удаляем старую копию репо если есть
REPO_DIR = "/content/colab-drive-transcriber"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("🗑️ Старая копия удалена")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# ── Шаг 1: whisperx без torch-зависимостей (используем Colab torch) ──
print("📦 Шаг 1/4: WhisperX (без torch)...")
pip("whisperx", "--no-deps")
pip("faster-whisper", "--no-deps")
pip("ctranslate2", "av", "tokenizers>=0.13")

# ── Шаг 2: pyannote (диаризация) ──
print("📦 Шаг 2/4: Pyannote...")
pip("pyannote.audio>=4.0")
pip("pyannote.core", "pyannote.database", "pyannote.metrics", "pyannote.pipeline")
pip("speechbrain", "asteroid-filterbanks", "torch-audiomentations", "einops", "lightning")

# ── Шаг 3: transformers (совместимая версия) ──
print("📦 Шаг 3/4: Transformers...")
pip("transformers>=4.40,<4.52", "huggingface-hub>=0.20")

# ── Шаг 4: утилиты ──
print("📦 Шаг 4/4: Утилиты...")
pip("noisereduce", "soundfile", "nltk", "pyyaml", "optuna", "hyperpyyaml", "docopt", "rich", "python-docx")

print("✅ Все пакеты установлены")

# ── Клонируем приватный репозиторий через GitHub Token ──
print("📥 Клонирование репозитория...")
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
clone_url = f"https://{github_token}@github.com/Hipposum/Hipposum-colab-drive-transcriber.git"
result = subprocess.run(
    ["git", "clone", "--depth", "1", clone_url, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("❌ Ошибка клонирования:", result.stderr)
    raise RuntimeError("Не удалось клонировать репозиторий. Проверь GITHUB_TOKEN в Secrets Colab.")
print("✅ Репозиторий клонирован")

## 📂 Google Drive и секреты

In [ ]:
import os
from google.colab import drive, userdata

# ── Подключаем Google Drive (разреши доступ во всплывающем окне) ──
drive.mount("/content/drive")

VIDEOS_DIR  = f"/content/drive/MyDrive/{DRIVE_FOLDER.strip('/')}"
RESULTS_DIR = f"{VIDEOS_DIR}/_results"

if not os.path.isdir(VIDEOS_DIR):
    raise RuntimeError(
        f"Папка {VIDEOS_DIR} не найдена. Создай папку '{DRIVE_FOLDER}' на Google Drive "
        "и положи туда видео (или поменяй DRIVE_FOLDER в настройках выше).")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── HF_TOKEN для диаризации ──
if DIARIZE:
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        raise RuntimeError("Добавь секрет HF_TOKEN (🔑 слева) и включи ему доступ из блокнота — "
                           "без него диаризация не работает. Либо поставь DIARIZE = False.")

# ── Передаём настройки пайплайну ──
env = {
    "PIPELINE_LOCAL_DIR":        VIDEOS_DIR,
    "PIPELINE_WORK_DIR":         RESULTS_DIR,          # результаты и чекпоинты — сразу на Drive
    "PIPELINE_DOWNLOAD_DIR":     "/content/videos_tmp",
    "PIPELINE_DIARIZE":          str(DIARIZE).lower(),
    "PIPELINE_MAX_VIDEOS":       str(MAX_VIDEOS),
    "PIPELINE_WHISPER_MODEL":    WHISPER_MODEL,
    "PIPELINE_LANGUAGE":         LANGUAGE,
    "PIPELINE_MIN_SPEAKERS":     str(MIN_SPEAKERS),
    "PIPELINE_MAX_SPEAKERS":     str(MAX_SPEAKERS),
    "PIPELINE_MIN_DURATION_SEC": str(MIN_DURATION_SEC),
}
if INITIAL_PROMPT.strip():
    env["PIPELINE_INITIAL_PROMPT"] = INITIAL_PROMPT.strip()
os.environ.update(env)

# ── Что нашли в папке ──
exts = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".zip"}
found = [f for root, _, files in os.walk(VIDEOS_DIR) if not root.startswith(RESULTS_DIR)
         for f in files if os.path.splitext(f)[-1].lower() in exts]
print(f"\n✅ Drive подключён. Видео в очереди: {len(found)}")
for f in sorted(found)[:20]:
    print(f"   • {f}")
if len(found) > 20:
    print(f"   ... и ещё {len(found) - 20}")
print(f"\nРезультаты будут в: {RESULTS_DIR}")

## 🚀 Запуск
Если сессия оборвётся — просто запусти всё заново: этапы кэшируются на Drive,
уже готовые видео пропускаются, недоделанные продолжатся с места остановки.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "src.runner"],
    cwd="/content/colab-drive-transcriber"
)
if result.returncode != 0:
    print(f"❌ Пайплайн завершился с ошибкой (код {result.returncode})")
else:
    print("\n🎉 Готово! Транскрипты — на Google Drive в папке _results "
          "(файл {имя_видео}.txt внутри папки с именем видео).")

## 📄 Что получается на выходе

В `Drive/<твоя папка>/_results/<имя видео>/`:

- **`<имя видео>.docx`** — красивый транскрипт для чтения (как у SpeechToText):
  «Спикер N:» жирным, под ним абзацы `ЧЧ:ММ:СС - текст`;
- `<имя видео>.txt` — тот же транскрипт в plain text;
- `<имя видео>_metrics.json` — метрики (длительность, баланс речи по спикерам, паузы);
- `<имя видео>_segments.json` — реплики с таймкодами и спикерами в JSON (удобно скармливать LLM).

**Частые вопросы:**
- *Диаризация упала с ошибкой доступа* → проверь, что принял условия моделей pyannote (ссылки в шапке) тем же аккаунтом, чей `HF_TOKEN`.
- *Надо быстрее и спикеры не нужны* → `DIARIZE = False`.
- *Обработать заново уже готовое видео* → удали его папку из `_results/` и строку с его именем из `_results/_progress.json`.
